#### 1. Installe Dependencies

#### 2. Configure API Keys

In [2]:
import os
from dotenv import load_dotenv

try:
    load_dotenv()
except Exception:
    pass

#### 3. Load Real Documents
#### We will create our private knowledge base from a real web document
#### "https://docs.langchain.com/oss/python/langgraph/agentic-rag"

In [3]:
from langchain_community.document_loaders import WebBaseLoader
Source_URL="https://docs.langchain.com/oss/python/langgraph/agentic-rag"

loader=WebBaseLoader(
    web_path=(Source_URL),
    requests_kwargs={
        "headers":{
            "User-Agent":"Mozilla/5.0 Agentic-RAG-Industry-Demo"
        }
    }
)
raw_docs = loader.load()
print("Loaded Documents: ", len(raw_docs))
print("Source: ", raw_docs[0].metadata.get("source"))
print("\nPreview:\n")
print(raw_docs[0].page_content[:1500])


C:\Users\MyPC\AppData\Local\Temp\ipykernel_9904\2651812096.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded Documents:  1
Source:  https://docs.langchain.com/oss/python/langgraph/agentic-rag

Preview:






Build a custom RAG agent with LangGraph - Docs by LangChain









Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewManaged Deep AgentsDeep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsData AnalysisDeep ResearchContent BuilderRAGLangChainDeep AgentSemantic searchSQL agentVoice agentMulti-agentSubagents: Personal assistantHandoffs: Customer supportRouter: Knowledge baseSkills: SQL assistantLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextHuman-in-the-loopGraph

#### 4. Split the documents into Chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True
)
chunks=splitter.split_documents(raw_docs)
print("Total Chunks:",len(chunks))
print("\nFirst chunk preview\n")
print(chunks[0].page_content[:900])

#### 5. Creating Free Local Embeddings

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings= HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings":True}
)
sample_vector=embeddings.embed_query("What is Agent Rag?")
print("Embedding dimentions: ", len(sample_vector))

#### 6. Creating the Pinecone Vector Database

In [ ]:
from pinecone import Pinecone,ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import time

INDEX_NAME="industry-agentic-rag-kb"
NAMESPACE="langgraph-agentic-rag"

#Connect to pinecone
pc=Pinecone()

#Create the index only if it does not already exist 
existing_indexes=[index_info["name"] for index_info in pc.list_indexes()]
if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

    #Wait until Pinecone reports the new index as ready
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)
print("Pinecone index is ready ",INDEX_NAME)

vectorstore=PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace=NAMESPACE
)

retriever=vectorstore.as_retriever(
    search_kwargs={
        "k":4,
        "namespace" :NAMESPACE
    }
)
print("Pinecone vector database and retriever are ready.")

In [24]:
#Load Existing Index
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

INDEX_NAME="industry-agentic-rag-kb"
NAMESPACE="langgraph-agentic-rag"

#Connect to pinecone
pc=Pinecone()

#Load Existing Pinecone Index
index=pc.Index(INDEX_NAME)

#Connect Existing Index with Langchain
vectorstore=PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace=NAMESPACE
)

#Create a retriever
retriever=vectorstore.as_retriever(
    search_kwargs={
        "k":4,
        "namespace":NAMESPACE
    }
)
print("Existing Pinecone Index Loaded: ",INDEX_NAME)

Existing Pinecone Index Loaded:  industry-agentic-rag-kb


#### 7. Retrieving the documents from Vector

In [ ]:
test_question="What happens if retrieved document are not relevant in Agentic RAG?"

kb_documents=retriever.invoke(test_question)

for i, doc in enumerate(kb_documents,1):
    print(f"\n---- KB Result {i} ---")
    print("Source: ", doc.metadata.get("source"))
    print(doc.page_content[:700])


#### 8. Initializing the Groq LLM

In [21]:
from langchain_groq import ChatGroq

llm=ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)
print(llm.invoke("Explain Agentic RAg in one sentense").content)

Agentic RAg is a retrieval‑augmented generation framework that lets an agent actively query, select, and incorporate external knowledge sources on the fly to guide and improve its text‑generation outputs.


#### 9. Initialize the Tavily Web Search
#### Tavily will be only used when the private knowledge base is not sufficient
#### IMPORTANT: Don't want to send every private/company question to the public web

In [23]:
from langchain_tavily import TavilySearch
websearch=TavilySearch(
    max_results=5,
    topic="general",
    include_answer=True,
    include_raw_content=False
)
print("Tavily search tool ready")

Tavily search tool ready


#### 10. Define Structured Decisions
#### We need reliable yes/no or route outputs from LLM.
#### Structured Outputs help avoid parsing messy natural language.

In [27]:
from typing import List,Literal,Optional
from typing_extensions import TypedDict
from pydantic import BaseModel,Field
from langchain_core.documents import Document

class RouteDecision(BaseModel):
    route:Literal["kb","direct"]=Field(
        description="Use kb for questions needing Agentic RAG docs; direct for greeting/simple chat."
    )

class EvidenceGrade(BaseModel):
    grade:Literal["good","weak"]=Field(
        description="good means evidence can the answer the question; weak means not enough evidence."
    )

class AgentState(TypedDict):
    question:str
    current_query:str
    kb_docs:List[Document]
    web_results:str
    kb_grade:str
    web_grade:str
    answer:str
    source_used:str
    retry_count:str

#### 11. Node 1 - Route the Question
#### The router decides:
####    . kb: use private knowledge base
####    . direct: answer without retrieval

In [28]:
router_llm = llm.with_structured_output(RouteDecision,method="json_mode")

def route_question(state:AgentState):
    question=state["question"]

    decision=router_llm.invoke(f'''
    You are a router for an Agentic Rag assistant.
    Route to "kb" if user ask about:
     - Agentic RAG
     - Langgraph Agentic RAG workflow
     - Retrieval grading
     - Query rewriting
     - RAG Architechture
     - Retriever tools
     - Webfallback in RAG

     Route to "direct" only for greetings, thanks, or very simple conversation.

     Question: {question}

    Return your response as valid JSON.
    Example:
    {{"route":"kb"}} ''')

    print("[Router]", decision.route)
    return{
        "current_query": question,
        "source_used":  decision.route
    }

def route_after_router(state:AgentState)-> Literal["retrieve_kb","direct_answer"]:
    if state["source_used"] =="kb":
        return "retrieve_kb"
    else:
        return "direct_answer"

#### 12. Node 2 - Retrieve from Private KB

In [29]:
def retrieve_kb(state:AgentState):
    query=state["current_query"]
    docs=retriever.invoke(query)

    print(f"[KB Retriever] Query: {query}")
    print(f"[KB Retriever] Retrieved: {len(docs)} chunks")

    return {"kb_docs":docs}


#### 13. Node 3 - Grade Private KB Evidence
#### This is the Key step that decides whether we can answer from the private KB or need web fallback

In [30]:
kb_grader_llm= llm.with_structured_output(EvidenceGrade,method="json_mode")

def grade_kb_evidence(state:AgentState):
    question=state["question"]
    context="\n\n".join(
        f"Source:{doc.metadata.get('source')}\n{doc.page_content}" 
        for doc in state["kb_docs"]
        )
    grade = kb_grader_llm.invoke(f'''
        You are an evidence grader.
        Question: {question}

        Private KB Evidence:
        {context}

        Can this private KB evidence answer the question?
        Return "good" if it can answer.
        Return "weak" if it can not give answer or incomplete.
        Return your response as Valid JSON
        Example:
        {{"grade":"good"}}
    ''')
    print("[KB Grader]",grade.grade)
    return {"kb_grade":grade.grade}

def decide_after_kb_grade(state:AgentState)-> Literal["generate_from_kb","search_web"]:
    if state["kb_grade"]=="good":
        return "generate_from_kb"
    else:
        return "search_web"
    

#### 14. Node 4 - Tavily Web Search Fallback
#### This node will run only if private KB evidence is weak.
#### We search the web usiing original question, or current rewritten query if a rewrite already happend

In [32]:
def search_web(state:AgentState):
    query=state["current_query"]
    print(f"[Travily Search] Query: {query}")
    result=websearch.invoke({"query":query})

    #Tavily can return dict/list structures depending on the package version
    #Convert it into a readable text for grading and generation
    if isinstance(result,dict):
        answer=result.get("answer","")
        results=result.get("results",[])
        lines=[]
        if answer:
            lines.append(f"Tavily answer: {answer}")

        for item in results:
            title=item.get("title","")
            url=item.get("url","")
            content=item.get("content","")
            lines.append(f"Title: {title}\nURL: {url}\nContent: {content}")
        web_text="\n\n".join(lines) if lines else str(result)
    else:
        web_text=str(result)
    print("[Tavily Search] Result characters: ",len(web_text))

    return {
        "web_results":web_text,
        "source_used": "web"
    }   

#### 15. Node 5 - Grade Web Evidence
#### This node will check the grade of web search whether web result is good or weak

In [33]:
web_grader_llm=llm.with_structured_output(EvidenceGrade,method="json_mode")

def grade_web_evidence(state:AgentState):
    question=state["question"]
    web_results=state["web_results"]

    grade=web_grader_llm.invoke(f'''
        You are an evidence grader.
        Question: {question}
        Web search evidence: {web_results}

        Can this web evidence answer the question?
        Return "good" if it can answer.
        Return "weak" if it can not give answer or incomplete.
        Return your response as Valid JSON
        Example:
            {{"grade":"good"}}
    ''')
    print(f"[Web Grader]", grade.grade)
    return {"web_grade":grade.grade}

MAX_RETRIES=1
def decide_after_web_grade(state:AgentState)-> Literal["generate_from_web","rewrite_query","answer_insufficient"]:
    if state["web_grade"]=="good":
        return "generate_from_web"
    
    if state["retry_count"] < MAX_RETRIES:
        return "rewrite_query"

    return "answer_insufficient"

#### 16. Node 6 - Rewrite the Query
#### If both KB and Web evidence are weak, we rewrite once and then try the KB again.
#### Why back to KB first?
#### Because the rewritten query may help retriver better private evidence before relying on web search again

In [35]:
from langchain_core.messages import HumanMessage

def  rewrite_query(state:AgentState):
    question=state["question"]
    retry_count=state["retry_count"]+1

    rewritten=llm.invoke(f'''
    Rewrite the question for better retrieval and web search.

    Rules:
        - Preserve original intent
        - Make it specific and search-friendly
        - Do not answer
        - Return only the re-written query
    
    Original Question:
        {question}        
    ''').content.strip()

    print("[Rewritten] query: ",rewritten)

    return {
        "current_query":rewritten,
        "retry_count":retry_count
    }

#### 17. Node 7 - Generate from Private KB

In [36]:
def generate_kb(state:AgentState):
    question=state["question"]
    context="\n\n".join(
        f"[KB Source: {doc.metadata.get("source")}]\n{doc.page_content}" 
        for doc in state["kb_docs"]
    )

    answer=llm.invoke(
        f'''
        You are a technical instructor.
        Answer using only from the private KB context.

        Rules:
        - Beginner-friendly explanation.
        - Do-not invent unsupported details.
        - Mention that the answer is based on the private KB.
        - Include source type: Private KB.

        Question:{question}
        Private KB Context:{context}
        '''
    ).content

    return {
        "answer":answer,
        "source_used": "private_kb"
    }

#### 18. Node 7 - Generate from Web Search

In [37]:
def generate_web(state:AgentState):
    question=state["question"]
    web_context=state["web_results"]

    answer=llm.invoke(
        f'''
        You are a technical instructor.
        The private KB was insufficient, So web search was used.
        Answer using only from the web context ONLY.

        Rules:
        - Beginner-friendly explanation.
        - Do-not invent unsupported details.
        - Mention that the answer is based on the Tavily Web search.
        - Include source type: Web Search.

        Question:{question}
        Private KB Context:{web_context}
        '''
    ).content

    return {
        "answer":answer,
        "source_used": "web_search"
    }

#### 19. Node 9 - Direct Answer

In [38]:
def direct_answer(state:AgentState):
    question=state["question"]
    
    answer=llm.invoke(
        f'''
        Respond briefly and naturally.
        
        Message:{question}
        '''
    ).content
    return {
            "answer":answer,
            "source_used": "direct"
        }


#### 20. Node 10 - Insufficient Evidence Answer

In [39]:
def answer_insufficient(static:AgentState):
    answer=(
        "I could not find enough reliable evidence in the private knowledge base"
        "or web search results to answer this confidently."
        "Please provide more specific documents or rephrase the question."
    )
    return {
            "answer":answer,
            "source_used": "insufficient_evidence"
            }

#### 21. Build the LangGraph Workflow